In [2]:
# 여러 사이트에서 긁어온 날짜 문자열이 제각각이다. 이를 YYYY-MM-DD 하나로 통일하는 함수를 작성하시오.
samples = [
    "2024.12.24",          "2024-12-24",        "2024/12/24",
    "24.12.24",            "2024년 12월 24일",   "2024년 3월 5일",
    "12/24/2024",          "2024.12.24 14:30",  "등록일 : 2024.12.24",
    "2024-13-45",          "작성일 없음",         "",
]
# 조건

# normalize_date(s) -> str | None 함수를 작성할 것
# 위 12가지를 모두 처리할 것 — 변환 불가능하면 None
# 두 자리 연도(24.12.24)는 2024로 해석할 것
# 월·일이 한 자리인 경우(3월 5일)도 03, 05로 채울 것
# 존재하지 않는 날짜(2024-13-45)는 None으로 처리할 것 — 정규표현식만으론 거를 수 없음. 후처리할 것
# 12/24/2024(미국식)와 2024/12/24를 구분할 것
# 각 입력에 대해 어떤 패턴으로 매치됐는지 함께 출력할 것 (기대결과 부분 확인)
# ※ 매칭된 것을 변수로써 꺼내는 방법 (명명 그룹)

In [ ]:
'''
기대 결과

2024.12.24            → 2024-12-24   [ymd_dot]
24.12.24              → 2024-12-24   [ymd_short]
2024년 3월 5일         → 2024-03-05   [ymd_kor]
12/24/2024            → 2024-12-24   [mdy_slash]
2024-13-45            → None         [ymd_dash · 유효하지 않은 날짜]
작성일 없음            → None         [매치 없음]
""                   → None         [빈 값]
'''

In [ ]:
import re

In [4]:
pattern=[
    ('ymd_kor',re.compile(r'(?P<y>\d{4})\s*년\s*(?P<m>\d{1,2})\s*월\s*(?P<d>\d{1,2})\s*일*')),
    ('ymd_sep',re.compile(r"(?P<y>\d{4})[.\-/](?P<m>\d{1,2})[.\-/](?P<d>\d{1,2})")),
    ('mdy_slash',re.compile(r"(?P<m>\d{1,2})/(?P<d>\d{1,2})/(?P<y>\d{4})")),
    ('ymd_short',re.compile(r"(?<!\d)(?P<y>\d{2})[.\-/](?P<m>\d{1,2})[.\-/](?P<d>\d{1,2})(?!\d)")),
]

In [ ]:
def normalize_date(s):
    if not s or not str(s).strip():
        return (None,'빈 값')

    for tag,patt in pattern:
        p=patt.search(str(s))
        if not p:
            continue

        y,m,d=int(p['y']),int(p['m']),int(p['d'])
        if tag=='ymd_short':
            y+=2000 # 24 -> 2024

        if d>31 or m>12:
            return (None, f'{tag} - 유효하지 않은 날짜')
        elif d>30 and m in (4,6,9,11):
            return (None, f'{tag} - 유효하지 않은 날짜')
        elif d>28 and m==2:
            return (None, f'{tag} - 유효하지 않은 날짜')
        else:
            return (f'{y}-{m}-{d}',tag)
    return (None, '매치 없음')

In [ ]:
for s in samples:
    value,tag=normalize_date(s)
    print(f'{s} -> {str(value)} [{tag}]')

2024.12.24 -> 2024-12-24 [ymd_sep]
2024-12-24 -> 2024-12-24 [ymd_sep]
2024/12/24 -> 2024-12-24 [ymd_sep]
24.12.24 -> 2024-12-24 [ymd_short]
2024년 12월 24일 -> 2024-12-24 [ymd_kor]
2024년 3월 5일 -> 2024-3-5 [ymd_kor]
12/24/2024 -> 2024-12-24 [mdy_slash]
2024.12.24 14:30 -> 2024-12-24 [ymd_sep]
등록일 : 2024.12.24 -> 2024-12-24 [ymd_sep]
2024-13-45 -> None [ymd_sep - 유효하지 않은 날짜]
작성일 없음 -> None [매치 없음]
 -> None [빈 값]


In [ ]:
# 웹 서버의 액세스 로그를 정규표현식으로 파싱해 분석하시오.

# 아래 내용을 담은 로그 파일 access.log 를 생성하시오.
'''
203.0.113.42 - - [06/Aug/2026:14:22:31 +0900] "GET /list?page=3 HTTP/1.1" 200 5321 "<https://example.com/>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
198.51.100.7 - - [06/Aug/2026:14:22:33 +0900] "POST /api/search HTTP/1.1" 429 118 "-" "python-requests/2.31.0"
203.0.113.42 - - [06/Aug/2026:14:22:35 +0900] "GET /detail/9981 HTTP/1.1" 404 209 "<https://example.com/list>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
'''
# 조건

# 하나의 정규표현식으로 한 줄에서 다음 7개를 추출할 것 
# ip / timestamp / method / path / status / bytes / user_agent

# 명명 그룹 (?P<name>...) 을 사용할 것

# 형식이 깨진 줄은 건너뛰고, 몇 줄을 건너뛰었는지 출력할 것

# 다음 세 가지를 집계해 출력할 것

# 상태코드별 요청 수

# 4xx·5xx가 발생한 경로 상위 5개

# 봇으로 의심되는 User-Agent 목록과 그 요청 수 (bot / crawler / spider / python-requests 포함 여부로 판정, 대소문자 무시)

# 결과를 access_report.csv로 저장할 것

In [ ]:
# 기대 결과
'''
파싱 3줄 · 건너뜀 0줄

── 상태코드별 요청 수 ──
status
200    1
404    1
429    1

── 4xx·5xx 발생 경로 상위 5 ──
path
/api/search     1
/detail/9981    1

── 봇 의심 User-Agent ──
user_agent
python-requests/2.31.0    1
  봇 요청 비율 33.3%'''

In [11]:
import re
import pandas as pd

In [12]:
log=re.compile(
    r'^(?P<ip>\d{1,3}(?:\.\d{1,3}){3})'
    r'\s+\S+\s+\S+\s+' # 중간 공백
    r'\[(?P<timestamp>[^\]]+)\]\s+'
    r'"(?P<method>[A-Z]+)\s+(?P<path>\S+)[^"]*"\s+'
    r'(?P<status>\d{3})\s+'
    r'(?P<bytes>\d+)\s+'                          
    r'"[^"]*"\s+' # referrer
    r'"(?P<user_agent>[^"]*)"'
)
bot=re.compile(
    'bot|crawler|spider|python-requests',re.IGNORECASE
)

In [23]:
def parse(path:str) -> tuple[pd.DataFrame,int]:
    rows=[]
    skipped=0
    # for line in path.readlines(): # AttributeError: 'str' object has no attribute 'readlines'
    with open(path,encoding='utf-8') as f:
        for line in f.readlines():
            line=line.strip()
            if not line:
                continue
            m=log.match(line)
            if not m:
                skipped+=1
                continue
            rows.append(m.groupdict())

    df=pd.DataFrame(rows)
    if len(df):
        df['status']=df['status'].astype(int)
        df['bytes']=df['bytes'].astype(int)
        # df['is_bot']=df['user-agent'].str.contains(bot)
        df["is_bot"] = df["user_agent"].map(
            lambda x: any(
                b in x.lower() for b in "bot|crawler|spider|python-requests".split('|')
            )
        )
    return df,skipped

In [28]:
df,skipped=parse('access.log')
print(f'파싱 {len(df)}줄, 건너뜀 {skipped}줄\n')
print('--- 상태코드별 요청 수 ---')
print(df['status'].value_counts().sort_index().to_string())
print('\n--- 4xx, 5xx 발생 경로 상위 5개 ---')
df_bad=df[df['status']>=400]
print(df_bad['path'].value_counts().head(5).to_string())
print('\n--- 봇 의심 User-Agent ---')
print(df[df['is_bot']]['user_agent'].value_counts().sort_index().to_string()
      if not df[df['is_bot']]['user_agent'].value_counts().empty else '없음')
print(f'    봇 요청 비율 {df['is_bot'].mean()*100:.2f}%')

파싱 3줄, 건너뜀 0줄

--- 상태코드별 요청 수 ---
status
200    1
404    1
429    1

--- 4xx, 5xx 발생 경로 상위 5개 ---
path
/api/search     1
/detail/9981    1

--- 봇 의심 User-Agent ---
user_agent
python-requests/2.31.0    1
    봇 요청 비율 33.33%


In [29]:
df.to_csv('access_report.csv',index=False,encoding='cp949')